# Paired virome: uninducible classifier (figure_s15 model)
Apply the trained figure_s15 inactive-virus classifier to six highly enriched
bulk↔virome pairs processed under:
`results/uhvdb_toolkit/toolkit/referenceanalyze/sracha_fastp_deacon_sylph_csvtk_seqkit_coverm_genecoverage`
**Ground truth (figure_s15 convention)**
- **TP** = bulk only (uninducible / inactive)
- **FP** = bulk + enriched (inducible / active)
- **FN** = enriched only
**Call rule:** predicted uninducible if `P(inactive) ≥ 0.92` (manuscript high-confidence cutoff).
Also computes closest-to-1 virus–host ratio (VHR) with GTDB r214 host matching.


In [1]:
### load packages
from __future__ import annotations
import math
from pathlib import Path
import joblib
import numpy as np
import pandas as pd
import polars as pl
from sklearn.metrics import (
    average_precision_score,
    confusion_matrix,
    matthews_corrcoef,
    precision_recall_fscore_support,
    roc_auc_score,
)
FIG_S15 = Path(".").resolve()
ROOT = FIG_S15.parents[1] if FIG_S15.name == "figure_s15" else Path("/mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB")
# Prefer running from figure_s15/; fall back to absolute paths
if not (FIG_S15 / "phage_activity_model_full.joblib").exists():
    FIG_S15 = ROOT / "uhvdb-manuscript" / "figure_s15"
    ROOT = FIG_S15.parents[1]
RESULT_DIR = (
    FIG_S15
    / "results/uhvdb_toolkit/toolkit/referenceanalyze"
    / "sracha_fastp_deacon_sylph_csvtk_seqkit_coverm_genecoverage"
)
MODEL_DIR = FIG_S15
METADATA = ROOT / "uhvdb-manuscript/figure_s18/uhvdb_v5_final_metadata_v2.tsv.gz"
FIG1 = ROOT / "uhvdb-manuscript/figure_1/uhvdb_human_metag_results/uhvdb_2026-03-26-2"
PHIST = FIG1 / "uhvdb_phisthost.tsv.gz"
CRISPR = FIG1 / "uhvdb_crisprhost.tsv.gz"
GTDB = ROOT / "uhvdb-manuscript/figure_4/sylph_tax/gtdb_r214_metadata.tsv.gz"
OUT_DIR = FIG_S15 / "paired_virome_results"
# Also mirror to the paired_virome_dataset2 sensitivity_results used by the canvas
MIRROR_DIR = ROOT / "uhvdb-manuscript-update/paired_virome_dataset2/sensitivity_results"
GENE_BREADTH_THRESHOLD = 0.8
INACTIVE_PROBABILITY_THRESHOLD = 0.92  # manuscript high-conf (~95% precision)
COMPLETED_GROUPS = ["H24", "H60", "NCF007", "S08", "NC159", "NC178"]
# Drop these from UHVDB metadata before joining gene-breadth so classifier
# features use coverage-based hallmarks (0–3), matching figure_s15 training.
META_HALLMARK_COLLISION_COLS = (
    "n_hallmarks",
    "mcp_hallmark",
    "terl_hallmark",
    "portal_hallmark",
)
UNINFORMATIVE_ANNOTATION_RE = (
    r"(?i)^\s*(?:hypothetical(?: protein)?|uncharacteri[sz]ed(?: protein)?|"
    r"unknown(?: protein| function)?|no annotation|no[_ -]?phrog|none|na|n/a|-)?\s*$"
)
OUT_DIR.mkdir(parents=True, exist_ok=True)
MIRROR_DIR.mkdir(parents=True, exist_ok=True)
print("RESULT_DIR:", RESULT_DIR)
print("OUT_DIR:", OUT_DIR)
print("model:", MODEL_DIR / "phage_activity_model_full.joblib")


RESULT_DIR: /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/results/uhvdb_toolkit/toolkit/referenceanalyze/sracha_fastp_deacon_sylph_csvtk_seqkit_coverm_genecoverage
OUT_DIR: /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results
model: /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/phage_activity_model_full.joblib


In [2]:
### Gene-breadth biological groups (shared with figure_s15)
def with_bio_group_flags(df: pl.DataFrame) -> pl.DataFrame:
    cols = df.columns
    prep = []
    for name in ("pharokka_category", "phold_category", "empathi_annot", "pharokka_annot"):
        if name in cols:
            prep.append(pl.col(name).fill_null(""))
        else:
            prep.append(pl.lit("").alias(name))
    out = df.with_columns(prep).with_columns(
        [
            pl.col("empathi_annot")
            .str.split("|")
            .list.get(0, null_on_oob=True)
            .fill_null("")
            .alias("empathi_token0"),
            pl.col("empathi_annot")
            .str.split("|")
            .list.get(1, null_on_oob=True)
            .fill_null("")
            .alias("empathi_token1"),
        ]
    )
    return (
        out.with_columns(
            [
                (
                    (pl.col("pharokka_category") == "head and packaging")
                    | pl.col("phold_category").str.contains("(?i)head|capsid|portal|terminase")
                    | (pl.col("empathi_token0") == "pvp")
                    | (pl.col("empathi_token0") == "packaging_assembly")
                    | pl.col("empathi_token1").is_in(
                        ["capsid", "terminase", "portal", "head-tail_joining"]
                    )
                ).alias("is_capsid_packaging"),
                (
                    (pl.col("pharokka_category") == "DNA, RNA and nucleotide metabolism")
                    | (pl.col("empathi_token0") == "DNA-associated")
                    | (pl.col("empathi_token0") == "RNA-associated")
                    | pl.col("empathi_token1").is_in(
                        ["nuclease", "annealing", "DNA_polymerase", "helicase"]
                    )
                ).alias("is_dna_metabolism"),
                (
                    (pl.col("pharokka_category") == "tail")
                    | pl.col("phold_category").str.contains(r"(?i)\btail\b")
                    | (pl.col("empathi_token1") == "tail")
                ).alias("is_tail"),
                (
                    (pl.col("pharokka_category") == "lysis")
                    | pl.col("phold_category").str.contains("(?i)lysis|holin|endolysin")
                    | (pl.col("empathi_token0") == "lysis")
                    | (pl.col("empathi_token0") == "cell_wall_depolymerase")
                    | pl.col("empathi_token1").is_in(["lysis", "holin"])
                ).alias("is_lysis"),
                (
                    (pl.col("pharokka_category") == "connector")
                    | pl.col("phold_category").str.contains("(?i)connector|head-tail|head–tail")
                ).alias("is_connector"),
                (
                    (pl.col("pharokka_category") == "transcription regulation")
                    | (pl.col("empathi_token0") == "transcriptional_regulator")
                    | (pl.col("empathi_token1") == "transcriptional_regulator")
                ).alias("is_transcription"),
                (
                    (pl.col("pharokka_category") == "integration and excision")
                    | pl.col("phold_category").str.contains("(?i)integrase|integration")
                    | (pl.col("empathi_token1") == "integration")
                ).alias("is_integration"),
                (
                    (
                        pl.col("pharokka_category")
                        == "moron, auxiliary metabolic gene and host takeover"
                    )
                    | pl.col("phold_category").str.contains(
                        r"(?i)auxiliary metabolic|host takeover|moron|anti[- ]?defen[cs]e"
                    )
                    | pl.col("empathi_token0").is_in(
                        ["amg", "auxiliary_metabolic", "host_takeover", "moron", "anti-defense"]
                    )
                    | pl.col("empathi_token1").is_in(
                        ["amg", "auxiliary_metabolic", "host_takeover", "moron", "anti-defense"]
                    )
                ).alias("is_amg_host_takeover"),
            ]
        )
        .with_columns(
            (
                pl.col("pharokka_category")
                .str.strip_chars()
                .str.to_lowercase()
                .is_in(["", "unknown function", "other"])
                & pl.col("pharokka_annot").str.contains(UNINFORMATIVE_ANNOTATION_RE)
                & pl.col("phold_category").str.contains(UNINFORMATIVE_ANNOTATION_RE)
                & pl.col("empathi_annot").str.contains(UNINFORMATIVE_ANNOTATION_RE)
            ).alias("is_unannotated")
        )
    )
def load_depth(path: Path, sample_id: str) -> pl.DataFrame:
    raw = pl.read_csv(path, separator="\t")
    cols = raw.columns
    rename = {
        cols[0]: "contig_id",
        cols[1]: "trimmed_mean",
        cols[2]: "mean",
        cols[3]: "variance",
        cols[4]: "covered_bases",
        cols[5]: "length",
    }
    group = sample_id.rsplit("_", 1)[0]
    return (
        raw.rename(rename)
        .with_columns(
            [
                pl.col("trimmed_mean").cast(pl.Float64),
                pl.col("mean").cast(pl.Float64),
                pl.col("variance").cast(pl.Float64),
                pl.col("covered_bases").cast(pl.Float64),
                pl.col("length").cast(pl.Float64),
                pl.lit(sample_id).alias("sample_id"),
                pl.lit(group).alias("group"),
                (pl.col("covered_bases") / pl.col("length")).alias("breadth"),
                (1 - math.e ** (-0.833 * pl.col("mean"))).alias("expected_breadth"),
            ]
        )
        .with_columns(
            pl.when(pl.col("expected_breadth") > 1e-6)
            .then(pl.col("breadth") / pl.col("expected_breadth"))
            .otherwise(None)
            .alias("breadth_ratio")
        )
    )
def load_gene_breadth(path: Path, sample_id: str) -> pl.DataFrame:
    df = (
        pl.read_csv(path, separator="\t")
        .filter(pl.col("breadth") > GENE_BREADTH_THRESHOLD)
        .with_columns(pl.lit(sample_id).alias("sample_id"))
    )
    return (
        with_bio_group_flags(df)
        .group_by(["sample_id", "genomovar_rep"])
        .agg(
            [
                pl.len().alias("n_genes_covered"),
                pl.col("is_capsid_packaging").mean().alias("prop_capsid_packaging"),
                pl.col("is_dna_metabolism").mean().alias("prop_dna_metabolism"),
                pl.col("is_tail").mean().alias("prop_tail"),
                pl.col("is_lysis").mean().alias("prop_lysis"),
                pl.col("is_connector").mean().alias("prop_connector"),
                pl.col("is_transcription").mean().alias("prop_transcription"),
                pl.col("is_integration").mean().alias("prop_integration"),
                pl.col("is_amg_host_takeover").mean().alias("prop_amg_host_takeover"),
                (
                    ((pl.col("pharokka_annot") == "major head protein").sum() >= 1)
                    | ((pl.col("phold_category") == "major head protein").sum() >= 1)
                    | ((pl.col("empathi_annot") == "pvp|capsid|major_capsid").sum() >= 1)
                )
                .cast(pl.UInt32)
                .alias("mcp_hallmark"),
                (
                    ((pl.col("pharokka_annot") == "terminase large subunit").sum() >= 1)
                    | ((pl.col("phold_category") == "terminase large subunit").sum() >= 1)
                    | (
                        (
                            pl.col("empathi_annot")
                            == "DNA-associated|terminase|packaging_assembly"
                        ).sum()
                        >= 1
                    )
                )
                .cast(pl.UInt32)
                .alias("terl_hallmark"),
                (
                    ((pl.col("pharokka_annot") == "portal protein").sum() >= 1)
                    | ((pl.col("phold_category") == "portal protein").sum() >= 1)
                    | ((pl.col("empathi_annot") == "pvp|portal").sum() >= 1)
                )
                .cast(pl.UInt32)
                .alias("portal_hallmark"),
            ]
        )
        .with_columns(
            (pl.col("mcp_hallmark") + pl.col("terl_hallmark") + pl.col("portal_hallmark")).alias(
                "n_hallmarks"
            )
        )
    )
def load_sylph_ani(path: Path, sample_id: str) -> pl.DataFrame:
    return (
        pl.read_csv(path, separator="\t")
        .filter(pl.col("Contig_name").str.starts_with("UHVDB-"))
        .with_columns(
            [
                pl.lit(sample_id).alias("sample_id"),
                pl.col("Contig_name").alias("contig_id"),
                pl.col("Adjusted_ANI").cast(pl.Float64).alias("ani"),
            ]
        )
        .select(["sample_id", "contig_id", "ani"])
    )
print("helpers ready")


helpers ready


In [3]:
### Discover completed bulk+enriched pairs and build feature table
available = {p.name.replace(".depth.tsv.gz", "") for p in RESULT_DIR.glob("*.depth.tsv.gz")}
pairs = []
for g in COMPLETED_GROUPS:
    bulk, enr = f"{g}_bulk", f"{g}_enriched"
    if bulk in available and enr in available:
        pairs.append(g)
if not pairs:
    raise SystemExit(f"No completed pairs found in {RESULT_DIR}")
print("Completed bulk+enriched pairs:", pairs)
uhvdb_cols = pl.read_csv(METADATA, separator="\t", n_rows=0).columns
uhvdb = pl.read_csv(METADATA, separator="\t").drop(
    [c for c in META_HALLMARK_COLLISION_COLS if c in uhvdb_cols]
)
depth_lst, gene_lst, ani_lst = [], [], []
for g in pairs:
    for kind in ("bulk", "enriched"):
        sid = f"{g}_{kind}"
        depth_lst.append(load_depth(RESULT_DIR / f"{sid}.depth.tsv.gz", sid))
        ani_lst.append(load_sylph_ani(RESULT_DIR / f"{sid}.profile.tsv", sid))
        # Gene-breadth features for classifier use bulk (unenriched) coverage,
        # matching figure_s15 training.
        if kind == "bulk":
            gene_lst.append(load_gene_breadth(RESULT_DIR / f"{sid}.gene_coverage.tsv.gz", sid))
coverm_df = pl.concat(depth_lst)
gene_breadth = pl.concat(gene_lst).rename({"genomovar_rep": "contig_id"})
ani_df = pl.concat(ani_lst)
joined = coverm_df.filter(pl.col("sample_id").str.contains("_bulk")).join(
    coverm_df.filter(pl.col("sample_id").str.contains("_enriched")),
    on=["group", "contig_id"],
    suffix="_enriched",
    how="full",
)
coalesce_exprs = []
if "contig_id_enriched" in joined.columns:
    coalesce_exprs.append(
        pl.coalesce([pl.col("contig_id"), pl.col("contig_id_enriched")]).alias("contig_id")
    )
if "group_enriched" in joined.columns:
    coalesce_exprs.append(
        pl.coalesce([pl.col("group"), pl.col("group_enriched")]).alias("group")
    )
if coalesce_exprs:
    joined = joined.with_columns(coalesce_exprs)
feat = (
    joined.filter(pl.col("contig_id").is_not_null())
    .join(uhvdb, left_on="contig_id", right_on="uhvdb_id", how="left")
    .filter(pl.col("seq_name") == pl.col("seqhash_rep"))
    .join(gene_breadth, on=["contig_id", "sample_id"], how="left")
    .join(ani_df, on=["sample_id", "contig_id"], how="left")
    .with_columns(
        [
            pl.coalesce([pl.col("sample_id"), pl.col("sample_id_enriched")]).alias("sample_id"),
            (pl.col("checkv_quality") == "Complete").cast(pl.Int64).alias("complete_count"),
            (pl.col("checkv_quality") == "High-quality")
            .cast(pl.Int64)
            .alias("high_quality_count"),
            pl.col("n_hallmarks").alias("med_n_hallmarks"),
            ((pl.col("aai_id") / 100) * pl.col("aai_af")).alias("med_aai_id_af"),
            pl.col("virulent").alias("med_virulent_score"),
            pl.col("prop_capsid_packaging").alias("med_prop_capsid_packaging"),
            pl.col("prop_dna_metabolism").alias("med_prop_dna_metabolism"),
            pl.col("prop_tail").alias("med_prop_tail"),
            pl.col("prop_lysis").alias("med_prop_lysis"),
            pl.col("prop_connector").alias("med_prop_connector"),
            pl.col("prop_transcription").alias("med_prop_transcription"),
            pl.col("prop_integration").alias("med_prop_integration"),
            pl.col("prop_amg_host_takeover").alias("med_prop_amg_host_takeover"),
            pl.col("mcp_hallmark").alias("med_mcp_hallmark"),
            pl.col("terl_hallmark").alias("med_terL_hallmark"),
            pl.col("portal_hallmark").alias("med_portal_hallmark"),
            pl.when(pl.col("trimmed_mean") > 0)
            .then(pl.col("variance") / pl.col("trimmed_mean"))
            .otherwise(None)
            .alias("variance_ratio"),
        ]
    )
    .with_columns(
        [
            pl.col(c).fill_null(0.0)
            for c in [
                "breadth",
                "breadth_enriched",
                "breadth_ratio",
                "breadth_ratio_enriched",
                "variance",
                "trimmed_mean",
                "complete_count",
                "high_quality_count",
            ]
        ]
    )
    .with_columns(
        pl.when((pl.col("breadth_ratio") > 0) & (pl.col("breadth_ratio_enriched") == 0))
        .then(pl.lit("TP"))
        .when((pl.col("breadth_ratio") == 0) & (pl.col("breadth_ratio_enriched") > 0))
        .then(pl.lit("FN"))
        .when((pl.col("breadth_ratio") > 0) & (pl.col("breadth_ratio_enriched") > 0))
        .then(pl.lit("FP"))
        .otherwise(pl.lit("TN"))
        .alias("pr_cat")
    )
    .with_columns(pl.col("species_cluster_id").cast(pl.Int64))
    .unique(["species_cluster_id", "sample_id"])
)
print("pr_cat counts:")
print(feat["pr_cat"].value_counts().sort("pr_cat"))


Completed bulk+enriched pairs: ['H24', 'H60', 'NCF007', 'S08', 'NC159', 'NC178']


pr_cat counts:
shape: (3, 2)
┌────────┬───────┐
│ pr_cat ┆ count │
│ ---    ┆ ---   │
│ str    ┆ u32   │
╞════════╪═══════╡
│ FN     ┆ 739   │
│ FP     ┆ 265   │
│ TP     ┆ 2421  │
└────────┴───────┘


In [4]:
### Apply figure_s15 inactive-virus classifier (p ≥ 0.92)
meta = joblib.load(MODEL_DIR / "phage_model_metadata_full.joblib")
pipeline = joblib.load(MODEL_DIR / "phage_activity_model_full.joblib")
numeric_cols = meta["numeric_cols"]
thresholds = {
    "high_95prec": meta["thresh_90"],
    "med_90prec": meta["thresh_75"],
    "low_85prec": meta["thresh_50"],
    "manuscript_0.92": INACTIVE_PROBABILITY_THRESHOLD,
}
print("numeric_cols:", numeric_cols)
print("thresholds:", thresholds)
bulk = feat.filter(pl.col("pr_cat").is_in(["TP", "FP"])).to_pandas()
y_true = (bulk["pr_cat"] == "TP").astype(int)  # positive = uninducible
X = bulk.reindex(columns=numeric_cols)
proba = pipeline.predict_proba(X)[:, 1]
bulk = bulk.copy()
bulk["predicted_inactive_probability"] = proba
bulk["y_true_uninducible"] = y_true
print("\nBulk-detected viruses:", len(bulk))
print("  True uninducible (TP):", int(y_true.sum()))
print("  True inducible (FP):", int((y_true == 0).sum()))
auroc = roc_auc_score(y_true, proba)
auprc = average_precision_score(y_true, proba)
print(f"  AUROC: {auroc:.3f}")
print(f"  AUPRC: {auprc:.3f}")
rows = []
for name, thr in thresholds.items():
    pred = (proba >= thr).astype(int)
    prec, rec, f1, _ = precision_recall_fscore_support(
        y_true, pred, average="binary", zero_division=0
    )
    tn, fp, fn, tp = confusion_matrix(y_true, pred, labels=[0, 1]).ravel()
    mcc = matthews_corrcoef(y_true, pred) if len(np.unique(pred)) > 1 else float("nan")
    rows.append(
        {
            "threshold_name": name,
            "threshold": thr,
            "n_bulk": len(bulk),
            "n_true_uninducible": int(y_true.sum()),
            "n_predicted_uninducible": int(pred.sum()),
            "TP": int(tp),
            "FP": int(fp),
            "FN": int(fn),
            "TN": int(tn),
            "precision": prec,
            "recall": rec,
            "f1": f1,
            "mcc": mcc,
            "auroc": auroc,
            "auprc": auprc,
        }
    )
    print(
        f"\n[{name}] p≥{thr:.4f}: "
        f"precision={prec:.3f} recall={rec:.3f} f1={f1:.3f} "
        f"pred={int(pred.sum())}/{len(bulk)} "
        f"(TP={tp} FP={fp} FN={fn} TN={tn})"
    )
metrics = pd.DataFrame(rows)
pred_default = (proba >= INACTIVE_PROBABILITY_THRESHOLD).astype(int)
bulk["predicted_uninducible"] = pred_default
by_group = (
    bulk.groupby("group", observed=True)
    .agg(
        n_bulk=("pr_cat", "size"),
        n_tp=("pr_cat", lambda s: int((s == "TP").sum())),
        n_fp=("pr_cat", lambda s: int((s == "FP").sum())),
        n_pred=("predicted_uninducible", "sum"),
        mean_prob=("predicted_inactive_probability", "mean"),
    )
    .reset_index()
)
pg_rows = []
for g, sub in bulk.groupby("group", observed=True):
    yt = sub["y_true_uninducible"].to_numpy()
    yp = sub["predicted_uninducible"].to_numpy()
    prec, rec, f1, _ = precision_recall_fscore_support(
        yt, yp, average="binary", zero_division=0
    )
    pg_rows.append({"group": g, "precision": prec, "recall": rec, "f1": f1})
by_group = by_group.merge(pd.DataFrame(pg_rows), on="group", how="left")
print("\nPer-group @ p≥0.92:")
print(by_group.to_string(index=False, float_format=lambda v: f"{v:.3f}"))
# Uninducible genome calls
uninducible = bulk.loc[bulk["predicted_uninducible"] == 1].copy()
print(f"\nPredicted uninducible genomes @ p≥0.92: {len(uninducible)}")
print(uninducible[["group", "contig_id", "species_cluster_id", "pr_cat", "predicted_inactive_probability"]]
      .sort_values(["group", "predicted_inactive_probability"], ascending=[True, False])
      .head(20)
      .to_string(index=False))


numeric_cols: ['breadth', 'breadth_ratio', 'variance_ratio', 'complete_count', 'high_quality_count', 'med_virulent_score', 'med_n_hallmarks', 'med_aai_id_af', 'med_prop_capsid_packaging', 'med_prop_dna_metabolism', 'med_prop_tail', 'med_prop_lysis', 'med_prop_connector', 'med_prop_transcription', 'med_prop_integration', 'med_prop_amg_host_takeover', 'med_mcp_hallmark', 'med_portal_hallmark', 'med_terL_hallmark', 'ani']
thresholds: {'high_95prec': 0.92, 'med_90prec': 0.81, 'low_85prec': 0.65, 'manuscript_0.92': 0.92}



Bulk-detected viruses: 2686
  True uninducible (TP): 2421
  True inducible (FP): 265
  AUROC: 0.674
  AUPRC: 0.939

[high_95prec] p≥0.9200: precision=0.939 recall=0.318 f1=0.475 pred=820/2686 (TP=770 FP=50 FN=1651 TN=215)

[med_90prec] p≥0.8100: precision=0.944 recall=0.587 f1=0.724 pred=1507/2686 (TP=1422 FP=85 FN=999 TN=180)

[low_85prec] p≥0.6500: precision=0.932 recall=0.824 f1=0.874 pred=2140/2686 (TP=1994 FP=146 FN=427 TN=119)

[manuscript_0.92] p≥0.9200: precision=0.939 recall=0.318 f1=0.475 pred=820/2686 (TP=770 FP=50 FN=1651 TN=215)

Per-group @ p≥0.92:
 group  n_bulk  n_tp  n_fp  n_pred  mean_prob  precision  recall    f1
   H24     461   360   101     149      0.780      0.906   0.375 0.530
   H60     199   143    56      74      0.811      0.824   0.427 0.562
 NC159     363   347    16     102      0.770      0.931   0.274 0.423
 NC178     591   543    48     176      0.785      0.977   0.317 0.478
NCF007     570   556    14     144      0.764      0.979   0.254 0.403
   S

In [5]:
### Save classifier outputs
feat_out = OUT_DIR / "paired_virome_classifier_features.tsv"
pred_out = OUT_DIR / "paired_virome_classifier_predictions.tsv"
metrics_out = OUT_DIR / "paired_virome_classifier_metrics.tsv"
group_out = OUT_DIR / "paired_virome_classifier_by_group.tsv"
unind_out = OUT_DIR / "paired_virome_uninducible_p0.92.tsv"
feat.write_csv(feat_out, separator="\t")
bulk.to_csv(pred_out, sep="\t", index=False)
metrics.to_csv(metrics_out, sep="\t", index=False)
by_group.to_csv(group_out, sep="\t", index=False)
uninducible.to_csv(unind_out, sep="\t", index=False)
for src in (feat_out, pred_out, metrics_out, group_out, unind_out):
    dst = MIRROR_DIR / src.name
    dst.write_bytes(src.read_bytes())
print("Wrote:")
for p in (feat_out, pred_out, metrics_out, group_out, unind_out):
    print(" ", p)


Wrote:
  /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_classifier_features.tsv
  /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_classifier_predictions.tsv
  /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_classifier_metrics.tsv
  /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_classifier_by_group.tsv
  /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_uninducible_p0.92.tsv


### Closest-to-1 VHR (GTDB host matching)
Primary figure_s15 VHR: among co-detected bacterial species in the predicted host genus
(PHIST+CRISPR majority), pick the host with phage/host abundance ratio nearest to 1.
Bacterial taxonomy from `Genome_file` GCA/GCF → GTDB r214 metadata.


In [6]:
### Build PHIST+CRISPR majority host table
def majority_host(df: pl.DataFrame, col: str) -> pl.DataFrame:
    return (
        df.filter(pl.col(col).is_not_null())
        .group_by(["species_cluster_id", col])
        .agg(pl.len().alias("n"))
        .sort(["species_cluster_id", "n", col], descending=[False, True, False])
        .unique("species_cluster_id", maintain_order=True)
        .select(["species_cluster_id", col])
    )
uhvdb_genomovar_host = (
    pl.read_csv(PHIST, separator="\t")[
        ["uhvdb_id", "total_connections", "rank", "agreement", "consensus_taxonomy"]
    ]
    .join(
        pl.read_csv(CRISPR, separator="\t")[
            ["uhvdb_id", "total_connections", "rank", "agreement", "top_taxonomy"]
        ],
        on=["uhvdb_id", "rank"],
        how="full",
        suffix="_crispr",
    )
    .fill_null(0)
    .with_columns(pl.col("consensus_taxonomy").str.replace(r"^(s|g|f)__", ""))
    .with_columns(
        [
            pl.when(pl.col("consensus_taxonomy") == pl.col("top_taxonomy"))
            .then(pl.col("consensus_taxonomy"))
            .when(
                (pl.col("total_connections") * pl.col("agreement"))
                >= (pl.col("total_connections_crispr") * pl.col("agreement_crispr"))
            )
            .then(pl.col("consensus_taxonomy"))
            .otherwise(pl.col("top_taxonomy"))
            .alias("final_taxonomy"),
            pl.coalesce(["rank", "rank_crispr"]).alias("final_rank"),
            pl.coalesce(["uhvdb_id", "uhvdb_id_crispr"]).alias("final_id"),
        ]
    )
    .group_by("final_id")
    .agg(
        [
            pl.col("final_taxonomy")
            .filter(pl.col("final_rank") == "species")
            .first()
            .alias("final_species"),
            pl.col("final_taxonomy")
            .filter(pl.col("final_rank") == "genus")
            .first()
            .alias("final_genus"),
            pl.col("final_taxonomy")
            .filter(pl.col("final_rank") == "family")
            .first()
            .alias("final_family"),
        ]
    )
    .rename({"final_id": "uhvdb_id"})
)
gv = (
    pl.read_csv(
        METADATA,
        separator="\t",
        columns=["uhvdb_id", "species_cluster_id", "genomovar_rep", "species_rep"],
    )
    .filter(pl.col("uhvdb_id") == pl.col("genomovar_rep"))
    .select(["uhvdb_id", "species_cluster_id", "species_rep"])
    .unique()
    .join(uhvdb_genomovar_host, on="uhvdb_id", how="left")
)
host = (
    gv.select(["species_cluster_id", "species_rep"])
    .unique()
    .join(majority_host(gv, "final_species"), on="species_cluster_id", how="left")
    .join(majority_host(gv, "final_genus"), on="species_cluster_id", how="left")
    .join(majority_host(gv, "final_family"), on="species_cluster_id", how="left")
)
print("host table species clusters:", host.height)


host table species clusters: 206289


In [7]:
### Compute closest-to-1 VHR for bulk samples
gtdb = (
    pl.read_csv(GTDB, separator="\t", has_header=False, new_columns=["accession", "lineage"])
    .with_columns(
        [
            pl.col("lineage").str.extract(r"g__([^;]+)", 1).alias("genus"),
            pl.col("lineage").str.extract(r"s__([^;]+)", 1).alias("species"),
            pl.col("lineage").str.extract(r"f__([^;]+)", 1).alias("family"),
        ]
    )
    .unique("accession")
)
id_map = (
    pl.read_csv(
        METADATA,
        separator="\t",
        columns=["uhvdb_id", "species_cluster_id", "seq_name", "seqhash_rep"],
    )
    .filter(pl.col("seq_name") == pl.col("seqhash_rep"))
    .select(["uhvdb_id", "species_cluster_id"])
    .unique()
)
virus_rows, bac_rows = [], []
for g in pairs:
    sid = f"{g}_bulk"
    df = pl.read_csv(RESULT_DIR / f"{sid}.profile.tsv", separator="\t")
    virus_rows.append(
        df.filter(pl.col("Contig_name").str.starts_with("UHVDB-"))
        .with_columns(
            [
                pl.lit(sid).alias("sample_id"),
                pl.col("Contig_name").alias("uhvdb_id"),
                pl.col("Taxonomic_abundance").cast(pl.Float64).alias("virus_tax_abund"),
            ]
        )
        .join(id_map, on="uhvdb_id", how="inner")
        .group_by(["sample_id", "species_cluster_id"])
        .agg(pl.col("virus_tax_abund").max())
    )
    bac = (
        df.filter(~pl.col("Contig_name").str.starts_with("UHVDB-"))
        .with_columns(
            [
                pl.lit(sid).alias("sample_id"),
                pl.col("Genome_file")
                .cast(pl.Utf8)
                .str.extract(r"(GC[AF]_\d+\.\d+)", 1)
                .alias("accession"),
                pl.col("Taxonomic_abundance").cast(pl.Float64).alias("host_tax_abund"),
            ]
        )
        .filter(pl.col("accession").is_not_null())
        .join(gtdb.select(["accession", "genus", "species", "family"]), on="accession", how="left")
        .filter(pl.col("genus").is_not_null())
        .group_by(["sample_id", "species", "genus", "family"])
        .agg(pl.col("host_tax_abund").sum())
    )
    bac_rows.append(bac)
viruses = (
    pl.concat(virus_rows)
    .join(
        host.select(["species_cluster_id", "final_species", "final_genus", "final_family"]),
        on="species_cluster_id",
        how="left",
    )
    .with_columns(pl.col("final_genus").alias("host_genus"))
)
bac_sylph = pl.concat(bac_rows)
vhr = (
    viruses.filter(pl.col("host_genus").is_not_null() & (pl.col("virus_tax_abund") > 0))
    .join(
        bac_sylph.select(["sample_id", "species", "genus", "host_tax_abund"]),
        left_on=["sample_id", "host_genus"],
        right_on=["sample_id", "genus"],
        how="inner",
    )
    .filter(pl.col("host_tax_abund") > 0)
    .with_columns(
        (pl.col("virus_tax_abund") / pl.col("host_tax_abund")).alias("phage_host_ratio")
    )
    .with_columns((pl.col("phage_host_ratio") - 1.0).abs().alias("dist_to_one"))
    .sort("dist_to_one")
    .unique(["sample_id", "species_cluster_id"], keep="first")
    .with_columns(
        [
            pl.col("sample_id").str.replace(r"_bulk$", "").alias("group"),
            pl.lit("closest_to_one").alias("host_match_method"),
            pl.col("species").alias("matched_host_species"),
        ]
    )
)
pred_pl = pl.from_pandas(bulk).with_columns(pl.col("species_cluster_id").cast(pl.Int64))
vhr_joined = (
    pred_pl.join(
        vhr.select(
            [
                "group",
                "species_cluster_id",
                "phage_host_ratio",
                "virus_tax_abund",
                "host_tax_abund",
                "host_genus",
                "matched_host_species",
                "host_match_method",
            ]
        ),
        on=["group", "species_cluster_id"],
        how="inner",
    )
    .select(
        [
            "group",
            "species_cluster_id",
            "pr_cat",
            "phage_host_ratio",
            "virus_tax_abund",
            "host_tax_abund",
            "host_genus",
            "matched_host_species",
            "predicted_inactive_probability",
        ]
    )
)
vhr_out = OUT_DIR / "paired_virome_vhr_gtdb_corrected.tsv"
vhr_joined.write_csv(vhr_out, separator="\t")
(MIRROR_DIR / vhr_out.name).write_bytes(vhr_out.read_bytes())
x = vhr_joined["phage_host_ratio"].to_numpy()
n_bulk = len(bulk)
print(f"Bulk with VHR: {vhr_joined.height}/{n_bulk} ({100 * vhr_joined.height / n_bulk:.1f}%)")
print(
    f"median={np.median(x):.3f} IQR=[{np.percentile(x, 25):.3f}, {np.percentile(x, 75):.3f}] "
    f"max={x.max():.1f}"
)
print(f"≥2: {(x >= 2).sum()} ({100 * (x >= 2).mean():.1f}%)  ≥8: {(x >= 8).sum()} ({100 * (x >= 8).mean():.1f}%)")
for t in [0, 1, 2, 4, 8, 16]:
    sub = vhr_joined if t == 0 else vhr_joined.filter(pl.col("phage_host_ratio") >= t)
    n = sub.height
    n_fp = sub.filter(pl.col("pr_cat") == "FP").height
    print(f"  VHR≥{t:g}: n={n} enriched={n_fp} ({100 * n_fp / n if n else float('nan'):.1f}%)")
print("\nVHR≥2 after uninducible filter:")
for thr in [0.92, 0.81, 0.65]:
    sub = vhr_joined.filter(
        (pl.col("phage_host_ratio") >= 2)
        & (pl.col("predicted_inactive_probability") < thr)
    )
    n = sub.height
    n_fp = sub.filter(pl.col("pr_cat") == "FP").height
    print(f"  after p≥{thr}: n={n} enriched%={100 * n_fp / n if n else float('nan'):.1f}")
print("Wrote", vhr_out)


Bulk with VHR: 2111/2686 (78.6%)
median=0.917 IQR=[0.618, 1.113] max=4682.7
≥2: 109 (5.2%)  ≥8: 33 (1.6%)
  VHR≥0: n=2111 enriched=218 (10.3%)
  VHR≥1: n=852 enriched=106 (12.4%)
  VHR≥2: n=109 enriched=21 (19.3%)
  VHR≥4: n=64 enriched=14 (21.9%)
  VHR≥8: n=33 enriched=10 (30.3%)
  VHR≥16: n=24 enriched=8 (33.3%)

VHR≥2 after uninducible filter:
  after p≥0.92: n=68 enriched%=23.5
  after p≥0.81: n=49 enriched%=30.6
  after p≥0.65: n=35 enriched%=37.1
Wrote /mmfs1/gscratch/pedslabs_hoffman/carsonjm/CFPhageome/repos/UHVDB/uhvdb-manuscript/figure_s15/paired_virome_results/paired_virome_vhr_gtdb_corrected.tsv


In [8]:
### Canvas summary JSON (detection + classifier + VHR)
summary = {
    "pairs": pairs,
    "detection": {
        "tp": int((feat["pr_cat"] == "TP").sum()),
        "fp": int((feat["pr_cat"] == "FP").sum()),
        "fn": int((feat["pr_cat"] == "FN").sum()),
        "bulk": int(len(bulk)),
    },
    "metrics": metrics.to_dict(orient="records"),
    "by_group": by_group.to_dict(orient="records"),
    "auroc": float(auroc),
    "auprc": float(auprc),
    "vhr": {
        "n_with_vhr": int(vhr_joined.height),
        "pct_bulk": float(100 * vhr_joined.height / n_bulk),
        "median": float(np.median(x)),
        "iqr": [float(np.percentile(x, 25)), float(np.percentile(x, 75))],
        "n_ge2": int((x >= 2).sum()),
        "pct_ge2": float(100 * (x >= 2).mean()),
    },
    "n_predicted_uninducible_p092": int(pred_default.sum()),
}
summary_path = OUT_DIR / "paired_virome_canvas_summary.json"
import json
summary_path.write_text(json.dumps(summary, indent=2))
(MIRROR_DIR / summary_path.name).write_text(summary_path.read_text())
print(json.dumps(summary, indent=2))


{
  "pairs": [
    "H24",
    "H60",
    "NCF007",
    "S08",
    "NC159",
    "NC178"
  ],
  "detection": {
    "tp": 2421,
    "fp": 265,
    "fn": 739,
    "bulk": 2686
  },
  "metrics": [
    {
      "threshold_name": "high_95prec",
      "threshold": 0.92,
      "n_bulk": 2686,
      "n_true_uninducible": 2421,
      "n_predicted_uninducible": 820,
      "TP": 770,
      "FP": 50,
      "FN": 1651,
      "TN": 215,
      "precision": 0.9390243902439024,
      "recall": 0.3180503923998348,
      "f1": 0.47516198704103674,
      "mcc": 0.08377125365258001,
      "auroc": 0.6743502217234418,
      "auprc": 0.93948514652343
    },
    {
      "threshold_name": "med_90prec",
      "threshold": 0.81,
      "n_bulk": 2686,
      "n_true_uninducible": 2421,
      "n_predicted_uninducible": 1507,
      "TP": 1422,
      "FP": 85,
      "FN": 999,
      "TN": 180,
      "precision": 0.9435965494359655,
      "recall": 0.587360594795539,
      "f1": 0.7240325865580448,
      "mcc": 0.1602051